# Likelihood and sampling with Firecrown

The `CLPFirecrown` stage writes the files CosmoSIS needs to sample the
likelihood
- `likelihood_file.py`, the [Firecrown](https://github.com/LSSTDESC/firecrown)
  likelihood,
- `priors_file.ini`, the values and priors of the parameters,
- `sampler_file.ini`, the configuration of CosmoSIS.

CosmoSIS then runs the chain, in a separate job.

Here we write these files for the cosmoDC2 redMaPPer baseline analysis, look at
what they contain, and evaluate the likelihood once. The inputs are in `data/`,
the data vector with its covariance written by `CLPCovariance`
(`clusters_sacc_file_cov.sacc`) and the cosmoDC2 fiducial cosmology
(`fiducial_cosmology.yml`).

Use the `firecrown_clp` kernel, as explained in [README.md](README.md).

In [1]:
import os
import shutil
import sys
sys.path.append("..")  # not needed if CLPipe is installed with `pip install -e .`

from clpipe.clp_firecrown import CLPFirecrown

## 1. Configuration

The options of the stage are in the `CLPFirecrown` section of the configuration
file. Here they are as a dictionary, with the choices of the baseline analysis.
The options that are not listed take the defaults of the stage, given in the
[documentation](https://lsstdesc.org/CLPipe/source/clp_firecrown.html).

In [2]:
config = {
    # Data used in the likelihood
    "use_cluster_counts": True,
    "use_shear_profile": True,
    "is_deltasigma": True,
    "use_mean_deltasigma": True,
    # Halo mass function, mass and redshift ranges
    "hmf": "despali16",
    "mass_def": "200c",
    "min_mass": 12.0,
    "max_mass": 15.5,
    "min_z": 0.2,
    "max_z": 0.8,
    # Selection function
    "use_completeness": True,
    "use_purity": False,
    # Sampler
    "sampler": "emcee",
    "emcee_walkers": 100,
    "emcee_samples": 50000,
    # Parameters

    "cosmological_parameters": {
        "omega_c": {"sample": True, "values": [0.1, 0.22, 0.5]},
        "sigma_8": {"sample": True, "values": [0.5, 0.8, 1.1]},
    },
    "firecrown_parameters": {
        "mass_distribution_mu0": {"sample": True, "values": [2.0, 3.3439, 10.0]},
        "mass_distribution_mu1": {"sample": True, "values": [0.5, 0.958236982, 2.0]},
        "mass_distribution_mu2": {"sample": True, "values": [-2.0, -0.0192802, 2.0]},
        "mass_distribution_sigma0": {"sample": True, "values": [0.1, 0.562317194, 2.0]},
        "mass_distribution_sigma1": {"sample": True, "values": [-0.6, 0.04552506, 0.3]},
        "mass_distribution_sigma2": {"sample": True, "values": [-0.5, -0.0445, 2.0]},
        "completeness_a_n": {"sample": False, "values": 1.1321},
        "completeness_b_n": {"sample": False, "values": 0.7751},
        "completeness_a_logm_piv": {"sample": False, "values": 13.31},
        "completeness_b_logm_piv": {"sample": False, "values": 0.2025},
        "cluster_theory_cluster_concentration": {"sample": False, "values": 3.8},
    },
}

Each parameter has a fixed value, or `[minimum, starting value, maximum]` when
it is sampled with a flat prior.
- `cosmological_parameters` uses the CosmoSIS names. It only needs the
  parameters that are sampled, the others take the value of the fiducial
  cosmology.
- `firecrown_parameters` has the parameters of the cluster model, the
  mass-richness relation (`mass_distribution_*`), the completeness
  (`completeness_*`) and the concentration of the halos.

## 2. Writing the files

In [3]:
os.makedirs("outputs", exist_ok=True)
# CosmoSIS runs from the folder of these files, and the likelihood opens the
# SACC file by its name, so the SACC file goes in the same folder.
shutil.copy("data/clusters_sacc_file_cov.sacc", "outputs/")

stage = CLPFirecrown.make_stage(
    clusters_sacc_file_cov="outputs/clusters_sacc_file_cov.sacc",
    fiducial_cosmology="data/fiducial_cosmology.yml",
    sampler_file="outputs/sampler_file.ini",
    likelihood_file="outputs/likelihood_file.py",
    priors_file="outputs/priors_file.ini",
    **config,
)
stage.run()

Python file generated at outputs/likelihood_file.py
INI file written to outputs/sampler_file.ini
Parameters INI file written to outputs/priors_file.ini


## 3. The likelihood

`likelihood_file.py` defines `build_likelihood`, the function Firecrown calls
to build the likelihood. The model is built with
[CROW](https://github.com/LSSTDESC/crow). The halo abundance and the
$\Delta\Sigma$ profile are integrated over mass and redshift by a recipe, with
the mass-richness relation and the completeness. The counts and the profiles
are two Firecrown statistics, combined in a Gaussian likelihood with the
covariance of the SACC file.

In [4]:
print(open("outputs/likelihood_file.py").read())

import os

import pyccl as ccl
import sacc

from crow import ClusterAbundance, ClusterShearProfile, kernel, mass_proxy
from crow.properties import ClusterProperty
from crow.recipes.binned_grid import GridBinnedClusterRecipe

from crow import purity_models, completeness_models
from firecrown.likelihood import (
    ConstGaussian,
    BinnedClusterShearProfile,
    BinnedClusterNumberCounts,
    Likelihood,
    NamedParameters,
)
from firecrown.modeling_tools import ModelingTools

def get_cluster_abundance() -> ClusterAbundance:
    """Creates and returns a ClusterAbundance object.""" 
    cluster_theory = ClusterAbundance(
    halo_mass_function = ccl.halos.MassFuncDespali16(mass_def="200c"),
    cosmo = ccl.CosmologyVanillaLCDM()
    )

    return cluster_theory


def get_cluster_shear_profile() -> ClusterShearProfile:
    """Creates and returns a ClusterShearProfile object."""
    cluster_theory = ClusterShearProfile(
    cosmo=ccl.CosmologyVanillaLCDM(),
    halo_mass_function = ccl.

## 4. The CosmoSIS files

`priors_file.ini` is the values file of CosmoSIS. It has every parameter of the
model, with the fiducial cosmology for the cosmological parameters that are not
sampled.

In [5]:
print(open("outputs/priors_file.ini").read())

[cosmological_parameters]
omega_c = 0.1 0.22 0.5
sigma_8 = 0.5 0.8 1.1
omega_b = 0.0448
h0 = 0.71
n_s = 0.963
omega_k = 0.0
w = -1.0
wa = 0.0
tau = 0.08
[firecrown_number_counts]
mass_distribution_mu0 = 2.0 3.3439 10.0
mass_distribution_mu1 = 0.5 0.958236982 2.0
mass_distribution_mu2 = -2.0 -0.0192802 2.0
mass_distribution_sigma0 = 0.1 0.562317194 2.0
mass_distribution_sigma1 = -0.6 0.04552506 0.3
mass_distribution_sigma2 = -0.5 -0.0445 2.0
completeness_a_n = 1.1321
completeness_b_n = 0.7751
completeness_a_logm_piv = 13.31
completeness_b_logm_piv = 0.2025
cluster_theory_cluster_concentration = 3.8



`sampler_file.ini` sets the sampler and the modules that compute the
likelihood. The `consistency` and `camb` modules compute the cosmology, and the
`firecrown_likelihood` module loads `likelihood_file.py`. The options of that
module (`use_cluster_counts`, ...) are passed to `build_likelihood`, and
`sampling_parameters_sections` names the section of the values file with the
parameters of the cluster model.

In [6]:
print(open("outputs/sampler_file.ini").read())

[runtime]
sampler = emcee
root = /sps/lsst/users/ebarroso/bla/CLPipe-tutorials/tutorials
resume = F

[default]
fatal_errors = F

[output]
filename = output_rp/number_counts_samples.txt
format = text
verbosity = 0

[pipeline]
modules = consistency camb firecrown_likelihood
values = priors_file.ini
likelihoods = firecrown
quiet = F
debug = F
timing = T

[consistency]
file = ${CSL_DIR}/utility/consistency/consistency_interface.py

[camb]
file = ${CSL_DIR}/boltzmann/camb/camb_interface.py

mode = all
lmax = 2500
feedback = 0
zmin = 0.0
zmax = 10.0
nz = 100
kmin = 1e-4
kmax = 50.0
nk = 1000

[firecrown_likelihood]
;; Fix this to use an environment variable to find the files.
;; Set FIRECROWN_DIR to the base of the firecrown installation (or build, if you haven't installed it)
file = /sps/lsst/groups/clusters/cl_pipeline_project/conda_envs/firecrown_developer_clp/lib/python3.13/site-packages/firecrown/connector/cosmosis/likelihood.py
likelihood_source = likelihood_file.py
sampling_parameters

## 5. Evaluating the likelihood once

Before submitting a chain, it is worth checking that the likelihood runs. The
`test` sampler of CosmoSIS evaluates it once, at the starting point, and saves
what it computed in the folder `output_counts_mean_mass`.

In [7]:
# CosmoSIS finds its standard library with CSL_DIR. In the shared environments
# it is installed in the environment itself.
os.environ.setdefault("CSL_DIR", os.path.join(sys.prefix, "cosmosis-standard-library"))
!cd outputs && cosmosis sampler_file.ini -p runtime.sampler=test

Deprecated: The [pipeline] quiet option is deprecated.  Set [runtime] verbosity instead.
Setting up pipeline from parameter file sampler_file.ini
--------------------------------------------------------

Setting up module consistency
------------------------------

Setting up module camb
-----------------------
/sps/lsst/groups/clusters/cl_pipeline_project/conda_envs/firecrown_developer_clp/cosmosis-standard-library/boltzmann/camb/camb_interface.py:183: UserWarning: Option kmin does not have an effect.
  warnings.warn("Option kmin does not have an effect.")
**** WARNING: Parameter 'kmin' in the [camb] section never used!


Setting up module firecrown_likelihood
---------------------------------------
/sps/lsst/users/ebarroso/crow/crow/cluster_modules/shear_profile.py:131: UserWarning: The concentration value '-1' results in theBhattacharya13 mass-concentration model being used.
  warnings.warn(
Setup all pipeline modules

Module timing:
consistency 0.001842
camb 0.002298
firecrown_like

The last lines give the logarithm of the prior, of the likelihood and of the
posterior. The log-likelihood is $-\chi^2/2$, so $\chi^2 = 246.1$ for 192 data
points. [theory_prediction.ipynb](theory_prediction.ipynb) shows how to compare
the model with the data at this point, or at any other.

## 6. Running the chain

In a run, the job script samples the likelihood from the output folder, with the
sampler of `sampler_file.ini`
```bash
mpirun -n ${SLURM_NTASKS} cosmosis --mpi sampler_file.ini
```
The chain is written to `output_rp/number_counts_samples.txt` (option
`filename`). [running_an_analysis.md](running_an_analysis.md) explains the job
scripts.